# Feature Engineering Practice : Ames Housing Data

### [Contents]  
- Ames Housing 데이터로 결측치, 이상치, 스케일링, 인코딩, 로그변환, 파생변수 생성을 실습
- 각 단계에서 학습자가 처리할 변수와 처리 방식을 직접 선택
- 최소한의 처리만 적용한 Baseline Model과 Feature Engineering 후 Final Model의 회귀 성능 비교
- 각 전처리 단계는 별도 복사본을 생성하므로 셀을 반복 실행해도 변환이 누적되지 않음
- 모든 전처리 기준은 Train 데이터에서만 학습(fit)하고 Test 데이터에는 동일한 기준을 적용(transform)

### [Feature Engineering Workflow]  
결측치 → 이상치 → 스케일링 → 인코딩 → 로그변환 → 파생변수 생성

### [Core Principle]  
Train에서 기준 학습(Fit) → Train 변환(Transform) → Test 동일 기준 적용(Transform)

### [Data]  
Ames Housing Raw 데이터를 바탕으로 교육용으로 간소화  
  
Target: SalePrice  
Numeric Features: 10개  
Categorical Features: 4개   

### [Note]  
  
배포된 "ames_housing_edu.csv" 파일을 직접 업로드  

## 1. 라이브러리 Import

- 데이터 처리: pandas, numpy
- 시각화: matplotlib, seaborn
- 회귀분석 및 전처리: scikit-learn


In [ ]:
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.compose import TransformedTargetRegressor

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid")


## 2. 데이터 업로드

- 배포된 "ames_housing_edu.csv" 파일을 업로드


In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise FileNotFoundError("업로드된 파일이 없습니다.")

csv_filename = next(iter(uploaded.keys()))
df_raw = pd.read_csv(io.BytesIO(uploaded[csv_filename]))

print("Uploaded file:", csv_filename)
print("Uploaded data shape:", df_raw.shape)
display(df_raw.head())


## 3. 데이터 구조 확인

[Question 1]  
- Target 변수와 설명변수는 무엇인가?
- 결측치가 있는 변수는 무엇인가?
- 수치형 변수와 범주형 변수는 각각 몇 개인가?
- 이후 단계에서 어떤 변수를 전처리해야 할 것으로 예상되는가?



In [ ]:
# 원본에서 항상 새 복사본을 생성
# 이 셀을 여러 번 실행해도 이전 전처리 결과가 누적되지 않음
df = df_raw.copy(deep=True)

target = "SalePrice"

if target not in df.columns:
    raise KeyError(
        f"{target} 컬럼이 없습니다. 현재 컬럼: {df.columns.tolist()}"
    )

X = df.drop(columns=[target]).copy()
y = df[target].copy()

numeric_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Target:", target)
print("Numeric variables:", len(numeric_columns), numeric_columns)
print("Categorical variables:", len(categorical_columns), categorical_columns)
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing_Count"))


## 4. 학습데이터와 평가데이터 분리

- 전처리 전에 Train/Test 데이터를 먼저 분리
- 결측치 대체값, 이상치 경계, 스케일링 통계량, 인코딩 범주는 Train에서만 학습
- Test 데이터에는 Train에서 학습한 기준을 그대로 적용
- Test 데이터의 분포와 통계량을 전처리 기준 계산에 사용하지 않음
- 원본 분할 결과를 별도 변수에 저장 (반복 실험시 초기화 고려)

In [ ]:
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 인덱스를 유지한 원본 복사본
X_train = X_train_raw.copy(deep=True)
X_test = X_test_raw.copy(deep=True)
y_train = y_train_raw.copy(deep=True)

print("Train data shape:", X_train.shape)
print("Test data shape :", X_test.shape)


## 5. Baseline 회귀분석

- 수치형 변수만 사용
- 결측치는 단순히 0으로 대체
- 이후 Final Model과 동일한 평가데이터로 성능 비교


In [ ]:
def evaluate_regression(y_true, y_pred, X_eval, model_name):
    r2 = r2_score(y_true, y_pred)

    n = X_eval.shape[0]  # 평가 샘플 수
    p = X_eval.shape[1]  # 독립변수 수

    if n <= p + 1:
        adjusted_r2 = np.nan
    else:
        adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    return pd.DataFrame({
        "Model": [model_name],
        "MAE": [mean_absolute_error(y_true, y_pred)],
        "RMSE": [np.sqrt(mean_squared_error(y_true, y_pred))],
        "Adjusted_R2": [adjusted_r2]
    })

baseline_columns = [
    "Overall_Qual",
    "Overall_Cond",
    "Gr_Liv_Area",
    "Lot_Area",
    "Year_Built",
    "Total_Bsmt_SF",
    "Garage_Cars",
    "Garage_Yr_Blt",
    "Full_Bath",
    "Bedroom_AbvGr"
]

X_train_base = X_train[baseline_columns].copy().fillna(0)
X_test_base = X_test[baseline_columns].copy().fillna(0)

baseline_model = LinearRegression()
baseline_model.fit(X_train_base, y_train)
baseline_pred = baseline_model.predict(X_test_base)

baseline_result = evaluate_regression(
    y_test,
    baseline_pred,
    X_test_base,
    "Baseline"
)

print("Baseline variables:", baseline_columns)
display(baseline_result.round(3))


## 6. Feature Engineering 실험 초기화

- 모든 전처리 실험은 원본 Train/Test 분할 결과에서 다시 시작
- 반복 실험할 때 이 셀부터 아래 셀을 순서대로 다시 실행


In [ ]:
X_train_step0 = X_train_raw.copy(deep=True)
X_test_step0 = X_test_raw.copy(deep=True)
y_train_step0 = y_train_raw.copy(deep=True)

print("Feature Engineering experiment reset.")
print("Train shape:", X_train_step0.shape)
print("Test shape :", X_test_step0.shape)


## 7. 결측치 처리

[Question 2]
- 결측치가 있는 변수는 무엇인가?
- 수치형 변수에는 평균, 중앙값, 0 중 어떤 방식이 적절한가?
- 범주형 변수에는 최빈값과 Missing 중 어떤 방식이 적절한가?

[Select]
- 수치형: median / mean / zero / none
- 범주형: most_frequent / missing / none

[Data Leakage Check]
- 대체 기준은 Train에서만 학습
- Train과 Test에 같은 기준 적용

In [ ]:
# [Select] 결측치 처리 방법
# 수치형: median / mean / zero / none
numeric_missing_settings = {
    "Garage_Yr_Blt": "median",
    "Total_Bsmt_SF": "median",
    "Garage_Cars": "median"
}

# 범주형: most_frequent / missing / none
categorical_missing_settings = {
    # "Neighborhood": "most_frequent",
    # "House_Style": "missing"
}

X_train_step1 = X_train_step0.copy(deep=True).replace({None: np.nan})
X_test_step1 = X_test_step0.copy(deep=True).replace({None: np.nan})
y_train_step1 = y_train_step0.copy(deep=True)

missing_summary = []
missing_imputers = {}

for column, method in numeric_missing_settings.items():
    if column not in X_train_step1.columns:
        print(f"Skip: {column} is not included in the data.")
        continue
    if method == "none":
        continue

    if method == "median":
        imputer = SimpleImputer(strategy="median")
    elif method == "mean":
        imputer = SimpleImputer(strategy="mean")
    elif method == "zero":
        imputer = SimpleImputer(strategy="constant", fill_value=0)
    else:
        raise ValueError(f"지원하지 않는 수치형 결측치 방법: {method}")

    missing_before = int(X_train_step1[column].isna().sum())
    imputer.fit(X_train_step1[[column]])
    X_train_step1[[column]] = imputer.transform(X_train_step1[[column]])
    X_test_step1[[column]] = imputer.transform(X_test_step1[[column]])

    missing_imputers[column] = imputer
    missing_summary.append({
        "Variable": column,
        "Method": method,
        "Train_Missing_Before": missing_before,
        "Train_Missing_After": int(X_train_step1[column].isna().sum())
    })

for column, method in categorical_missing_settings.items():
    if column not in X_train_step1.columns:
        print(f"Skip: {column} is not included in the data.")
        continue
    if method == "none":
        continue

    if method == "most_frequent":
        imputer = SimpleImputer(strategy="most_frequent")
    elif method == "missing":
        imputer = SimpleImputer(strategy="constant", fill_value="Missing")
    else:
        raise ValueError(f"지원하지 않는 범주형 결측치 방법: {method}")

    missing_before = int(X_train_step1[column].isna().sum())
    imputer.fit(X_train_step1[[column]])
    X_train_step1[[column]] = imputer.transform(X_train_step1[[column]])
    X_test_step1[[column]] = imputer.transform(X_test_step1[[column]])

    missing_imputers[column] = imputer
    missing_summary.append({
        "Variable": column,
        "Method": method,
        "Train_Missing_Before": missing_before,
        "Train_Missing_After": int(X_train_step1[column].isna().sum())
    })

display(pd.DataFrame(missing_summary))

## 8. 이상치 처리

[Question 3]
- 이상치를 제거하면 학습데이터 수는 어떻게 달라지는가?
- Clip은 이상치를 어떤 값으로 바꾸는가?
- 실제 대형 주택을 이상치로 처리해도 되는가?

[Select]
- 변수별: remove / clip / none

[Data Leakage Check]
- IQR 경계는 Train에서만 계산
- remove는 Train에만 적용
- clip은 Train 기준을 Train과 Test에 동일 적용

In [ ]:
# [Select] 변수별 이상치 처리 방법
# 선택 가능: remove / clip / none
outlier_settings = {
    "Gr_Liv_Area": "clip",
    "Lot_Area": "clip",
    "Total_Bsmt_SF": "none"
}

# IQR 배수
iqr_multiplier = 1.5

X_train_step2 = X_train_step1.copy(deep=True)
X_test_step2 = X_test_step1.copy(deep=True)
y_train_step2 = y_train_step1.copy(deep=True)

outlier_summary = []
outlier_bounds = {}
remove_mask = pd.Series(False, index=X_train_step2.index)

for column, method in outlier_settings.items():
    if column not in X_train_step2.columns:
        print(f"Skip: {column} is not included in the data.")
        continue

    if method == "none":
        continue

    if not pd.api.types.is_numeric_dtype(X_train_step2[column]):
        print(f"Skip: {column} is not numeric.")
        continue

    q1 = X_train_step2[column].quantile(0.25)
    q3 = X_train_step2[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - iqr_multiplier * iqr
    upper_bound = q3 + iqr_multiplier * iqr

    outlier_bounds[column] = {
        "lower": lower_bound,
        "upper": upper_bound
    }

    train_outlier_mask = (
        (X_train_step2[column] < lower_bound)
        | (X_train_step2[column] > upper_bound)
    )
    outlier_count = int(train_outlier_mask.sum())

    if method == "remove":
        remove_mask = remove_mask | train_outlier_mask

    elif method == "clip":
        X_train_step2[column] = X_train_step2[column].clip(
            lower=lower_bound,
            upper=upper_bound
        )
        X_test_step2[column] = X_test_step2[column].clip(
            lower=lower_bound,
            upper=upper_bound
        )

    else:
        raise ValueError(f"지원하지 않는 이상치 처리 방법: {method}")

    outlier_summary.append({
        "Variable": column,
        "Method": method,
        "Train_Q1": q1,
        "Train_Q3": q3,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Train_Outlier_Count": outlier_count
    })

# Remove를 선택한 이상치 행은 Train과 y_train에서만 함께 제거
removed_count = int(remove_mask.sum())
if removed_count > 0:
    keep_mask = ~remove_mask
    X_train_step2 = X_train_step2.loc[keep_mask].copy()
    y_train_step2 = y_train_step2.loc[keep_mask].copy()

print("Removed train rows:", removed_count)
print("Train shape after outlier handling:", X_train_step2.shape)
print("Test shape remains unchanged       :", X_test_step2.shape)
display(pd.DataFrame(outlier_summary).round(2))


## 9. 스케일링

[Question 4]
- 변수마다 단위와 범위가 다른가?
- Standard, Robust, Min-Max Scaling은 어떻게 다른가?
- 스케일링은 회귀계수 비교에 어떤 영향을 주는가?

[Select]
- scaling_method=standard / robust / minmax / none
- scaling_columns에 적용 변수 지정

[Data Leakage Check]
- Scaler는 Train에서만 fit
- Train과 Test에는 같은 Scaler로 transform

[Note]
- 로그변환할 변수는 스케일링 대상에서 제외 권장

In [ ]:
# [Select] 스케일링 방법과 변수
# scaling_method=standard / robust / minmax / none
scaling_method = "standard"

# scaling_columns에 적용 변수 지정
scaling_columns = [
    "Overall_Qual",
    "Overall_Cond",
    "Year_Built",
    "Garage_Cars",
    "Full_Bath",
    "Bedroom_AbvGr"
]

X_train_step3 = X_train_step2.copy(deep=True)
X_test_step3 = X_test_step2.copy(deep=True)
y_train_step3 = y_train_step2.copy(deep=True)

valid_scaling_columns = [
    column for column in scaling_columns
    if column in X_train_step3.columns
    and pd.api.types.is_numeric_dtype(X_train_step3[column])
]

if scaling_method == "standard":
    scaler = StandardScaler()
elif scaling_method == "robust":
    scaler = RobustScaler()
elif scaling_method == "minmax":
    scaler = MinMaxScaler()
elif scaling_method == "none":
    scaler = None
else:
    raise ValueError(
        "scaling_method는 'standard', 'robust', 'minmax', 'none' 중 선택하세요."
    )

scaling_fitted_statistics = {}

if scaler is not None and len(valid_scaling_columns) > 0:
    scaler.fit(X_train_step3[valid_scaling_columns])

    X_train_step3[valid_scaling_columns] = scaler.transform(
        X_train_step3[valid_scaling_columns]
    )
    X_test_step3[valid_scaling_columns] = scaler.transform(
        X_test_step3[valid_scaling_columns]
    )

    if scaling_method == "standard":
        scaling_fitted_statistics = {
            "center": scaler.mean_,
            "scale": scaler.scale_
        }
    elif scaling_method == "robust":
        scaling_fitted_statistics = {
            "center": scaler.center_,
            "scale": scaler.scale_
        }
    elif scaling_method == "minmax":
        scaling_fitted_statistics = {
            "data_min": scaler.data_min_,
            "data_max": scaler.data_max_
        }

print("Scaling method:", scaling_method)
print("Scaled variables:", valid_scaling_columns)
print("Scaler fitted on Train only:", scaler is not None)


## 10. 범주형 변수 인코딩

[Question 5]
- 순서가 없는 범주에 숫자를 부여하면 어떤 문제가 생기는가?
- One-Hot Encoding과 Dummy Coding의 변수 수는 어떻게 다른가?
- Label Encoding은 어떤 변수에 적절한가?

[Select]
- 변수별: label / onehot / dummy / none

[Data Leakage Check]
- Encoder는 Train에서만 fit
- Test의 새로운 범주는 무시하거나 -1 처리
- Train과 Test의 최종 열 구조 유지

In [ ]:
# [Select] 변수별 인코딩 방법
# 선택 가능: label / onehot / dummy / none
encoding_settings = {
    "Neighborhood": "onehot",
    "House_Style": "dummy"
}

X_train_step4 = X_train_step3.copy(deep=True)
X_test_step4 = X_test_step3.copy(deep=True)
y_train_step4 = y_train_step3.copy(deep=True)

encoding_summary = []
encoding_objects = {}

for column, method in encoding_settings.items():
    if column not in X_train_step4.columns:
        print(f"Skip: {column} is not included in the data.")
        continue

    if method == "none":
        continue

    if method == "label":
        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
        encoder.fit(X_train_step4[[column]].astype(str))

        X_train_step4[[column]] = encoder.transform(
            X_train_step4[[column]].astype(str)
        )
        X_test_step4[[column]] = encoder.transform(
            X_test_step4[[column]].astype(str)
        )

        encoding_objects[column] = encoder
        created_count = 1

    elif method in ["onehot", "dummy"]:
        drop_option = "first" if method == "dummy" else None
        encoder = OneHotEncoder(
            drop=drop_option,
            handle_unknown="ignore",
            sparse_output=False,
            dtype=int
        )

        encoder.fit(X_train_step4[[column]].astype(str))

        train_array = encoder.transform(
            X_train_step4[[column]].astype(str)
        )
        test_array = encoder.transform(
            X_test_step4[[column]].astype(str)
        )

        encoded_columns = encoder.get_feature_names_out([column])

        train_encoded = pd.DataFrame(
            train_array,
            columns=encoded_columns,
            index=X_train_step4.index
        )
        test_encoded = pd.DataFrame(
            test_array,
            columns=encoded_columns,
            index=X_test_step4.index
        )

        X_train_step4 = pd.concat(
            [X_train_step4.drop(columns=[column]), train_encoded],
            axis=1
        )
        X_test_step4 = pd.concat(
            [X_test_step4.drop(columns=[column]), test_encoded],
            axis=1
        )

        encoding_objects[column] = encoder
        created_count = len(encoded_columns)

    else:
        raise ValueError(f"지원하지 않는 인코딩 방법: {method}")

    encoding_summary.append({
        "Variable": column,
        "Method": method,
        "Train_Fitted_Categories": len(encoding_objects[column].categories_[0]),
        "Created_Columns": created_count
    })

# 선택하지 않은 문자열 변수가 남으면 모델 실행을 위해 자동 Dummy Coding
remaining_categorical = X_train_step4.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for column in remaining_categorical:
    encoder = OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False,
        dtype=int
    )
    encoder.fit(X_train_step4[[column]].astype(str))

    train_array = encoder.transform(X_train_step4[[column]].astype(str))
    test_array = encoder.transform(X_test_step4[[column]].astype(str))
    encoded_columns = encoder.get_feature_names_out([column])

    train_encoded = pd.DataFrame(
        train_array,
        columns=encoded_columns,
        index=X_train_step4.index
    )
    test_encoded = pd.DataFrame(
        test_array,
        columns=encoded_columns,
        index=X_test_step4.index
    )

    X_train_step4 = pd.concat(
        [X_train_step4.drop(columns=[column]), train_encoded],
        axis=1
    )
    X_test_step4 = pd.concat(
        [X_test_step4.drop(columns=[column]), test_encoded],
        axis=1
    )

    encoding_objects[column] = encoder
    encoding_summary.append({
        "Variable": column,
        "Method": "auto_dummy",
        "Train_Fitted_Categories": len(encoder.categories_[0]),
        "Created_Columns": len(encoded_columns)
    })

# 최종적으로 Train 기준 열 순서에 Test를 맞춤
X_test_step4 = X_test_step4.reindex(
    columns=X_train_step4.columns,
    fill_value=0
)

print("Feature count after encoding:", X_train_step4.shape[1])
print("Train/Test columns identical:", X_train_step4.columns.equals(X_test_step4.columns))
display(pd.DataFrame(encoding_summary))


## 11. 로그변환

[Question 6]
- 오른쪽으로 긴 꼬리를 가진 변수는 무엇인가?
- 로그변환 후 왜도는 어떻게 달라지는가?
- 0이 포함된 변수에 log1p()를 사용하는 이유는 무엇인가?

[Select]
- 변수별: log1p / none
- log_target=True / False

[Note]

In [ ]:
# [Select] 로그변환 방법
# 선택 가능: log1p / none
log_settings = {
    "Lot_Area": "log1p",
    "Gr_Liv_Area": "log1p",
    "Total_Bsmt_SF": "log1p"
}

# log_target=True / False
log_target = True

X_train_step5 = X_train_step4.copy(deep=True)
X_test_step5 = X_test_step4.copy(deep=True)
y_train_step5 = y_train_step4.copy(deep=True)

log_summary = []

for column, method in log_settings.items():
    if column not in X_train_step5.columns:
        print(f"Skip: {column} is not included in the data.")
        continue

    if method == "none":
        continue
    elif method != "log1p":
        raise ValueError(f"지원하지 않는 로그변환 방법: {method}")

    if X_train_step5[column].min() < -1 or X_test_step5[column].min() < -1:
        print(f"Skip: {column} contains values below -1 ")
        continue

    skew_before = X_train_step5[column].skew()
    X_train_step5[column] = np.log1p(X_train_step5[column])
    X_test_step5[column] = np.log1p(X_test_step5[column])
    skew_after = X_train_step5[column].skew()

    log_summary.append({
        "Variable": column,
        "Method": method,
        "Skew_Before": skew_before,
        "Skew_After": skew_after
    })

display(pd.DataFrame(log_summary).round(3))
print("Log transform target:", log_target)


## 12. 파생변수 생성

[Mission]
- EDA 결과를 바탕으로 파생변수를 설계해보세요
- Train과 Test에 같은 계산식을 적용
- 아무 변수도 만들지 않아도 다음 단계는 실행됨

[Example]
- Total_Area: 전체 면적
- Quality_Area: 품질과 면적의 상호작용
- House_Age: 주택 연식
- Area_per_Bedroom: 침실당 면적

In [ ]:
# 이전 단계에서 복사본 생성
X_train_final = X_train_step5.copy(deep=True)
X_test_final = X_test_step5.copy(deep=True)
y_train_final = y_train_step5.copy(deep=True)

base_columns = X_train_final.columns.tolist()

# ==========================================
# [Practice] 새로운 파생변수를 작성하세요.
# ==========================================
# 작성 원칙
# 1. Train과 Test에 같은 변수명과 같은 계산식을 적용합니다.
# 2. Target인 SalePrice는 파생변수 계산에 사용하지 않습니다.
# 3. 0으로 나눌 가능성이 있으면 분모에 작은 값 또는 조건문을 사용합니다. [에러 발생하므로 주의]
# 4. 아무 코드도 작성하지 않아도 이후 단계는 정상적으로 실행됩니다.

# 아래에 신규 파생 변수를 작성하세요.



# ==========================================
# [Example] 참고용 예시
# 사용할 예시의 주석을 해제하거나 새로운 식으로 수정하세요.
# ==========================================

# 예시 1. 전체 사용 면적
# X_train_final["Total_Area"] = (
#     X_train_final["Gr_Liv_Area"]
#     + X_train_final["Total_Bsmt_SF"]
# )
# X_test_final["Total_Area"] = (
#     X_test_final["Gr_Liv_Area"]
#     + X_test_final["Total_Bsmt_SF"]
# )

# 예시 2. 품질과 면적의 결합
# X_train_final["Quality_Area"] = (
#     X_train_final["Overall_Qual"]
#     * X_train_final["Gr_Liv_Area"]
# )
# X_test_final["Quality_Area"] = (
#     X_test_final["Overall_Qual"]
#     * X_test_final["Gr_Liv_Area"]
# )

# 예시 3. 주택 연식
# 판매연도가 데이터에 없으므로 교육용 기준연도 2010을 사용
# X_train_final["House_Age_2010"] = (
#     2010 - X_train_final["Year_Built"]
# )
# X_test_final["House_Age_2010"] = (
#     2010 - X_test_final["Year_Built"]
# )

# 예시 4. 침실당 지상 생활면적
# 분모가 0인 경우를 방지하기 위해 replace(0, 1)을 사용
# X_train_final["Area_per_Bedroom"] = (
#     X_train_final["Gr_Liv_Area"]
#     / X_train_final["Bedroom_AbvGr"].replace(0, 1)
# )
# X_test_final["Area_per_Bedroom"] = (
#     X_test_final["Gr_Liv_Area"]
#     / X_test_final["Bedroom_AbvGr"].replace(0, 1)
# )

# 예시 5. 차고 활용 점수
# X_train_final["Garage_Score"] = (
#     X_train_final["Garage_Cars"]
#     * X_train_final["Garage_Yr_Blt"]
# )
# X_test_final["Garage_Score"] = (
#     X_test_final["Garage_Cars"]
#     * X_test_final["Garage_Yr_Blt"]
# )

# ==========================================
# 생성 결과 확인
# ==========================================
created_train = [
    column for column in X_train_final.columns
    if column not in base_columns
]
created_test = [
    column for column in X_test_final.columns
    if column not in base_columns
]

# Train과 Test에 같은 파생변수가 생성되었는지 확인
if set(created_train) != set(created_test):
    raise ValueError(
        "Train과 Test에 생성된 파생변수가 다름 "
        f"Train: {created_train}, Test: {created_test}"
    )

# 이후 셀에서 사용할 파생변수 목록
created_features = created_train

print("Created features:", created_features)
print("Final train shape:", X_train_final.shape)
print("Final test shape :", X_test_final.shape)

if len(created_features) == 0:
    print("No derived features were created. ")
else:
    display(X_train_final[created_features].head())


## 13. 전처리 결과 확인

[Question 8]  
- 결측치는 모두 처리되었는가?
- 이상치 제거 시 학습데이터 수는 얼마나 감소했는가?
- 인코딩 후 변수 수는 어떻게 달라졌는가?
- 파생변수를 생성하지 않아도 데이터가 정상적으로 준비되었는가?


In [ ]:
print("Train data shape:", X_train_final.shape)
print("Test data shape :", X_test_final.shape)
print("Train target shape:", y_train_final.shape)

print()
print("Train missing values:", X_train_final.isna().sum().sum())
print("Test missing values :", X_test_final.isna().sum().sum())
print("Created features:", created_features)
print("Train/Test columns identical:", X_train_final.columns.equals(X_test_final.columns))

non_numeric = X_train_final.select_dtypes(exclude=np.number).columns.tolist()
print("Remaining non-numeric variables:", non_numeric)

if X_train_final.isna().sum().sum() > 0 or X_test_final.isna().sum().sum() > 0:
    raise ValueError("전처리 후 결측치가 남아 있습니다.")

if len(non_numeric) > 0:
    raise ValueError(f"인코딩되지 않은 범주형 변수가 남아 있습니다: {non_numeric}")

if not X_train_final.columns.equals(X_test_final.columns):
    raise ValueError("Train과 Test의 최종 변수 구조가 일치하지 않습니다.")


## 14. Final 회귀분석

- 모든 전처리가 적용된 데이터로 Linear Regression 모델 생성
- log_target=True이면 Target을 로그변환하고 예측값은 원래 단위로 복원
- Baseline과 같은 Test 데이터로 성능 비교

In [ ]:
final_regressor = LinearRegression()

if log_target:
    final_model = TransformedTargetRegressor(
        regressor=final_regressor,
        func=np.log1p,
        inverse_func=np.expm1
    )
else:
    final_model = final_regressor

final_model.fit(X_train_final, y_train_final)
final_pred = final_model.predict(X_test_final)

final_result = evaluate_regression(
    y_test,
    final_pred,
    X_test_final,
    "Feature Engineering"
)

display(final_result.round(3))


## 15. 회귀분석 성과 비교

[Question 9]  
- MAE와 RMSE는 감소했는가?
- Adjusted R²는 증가했는가?
- 어떤 전처리가 성능 변화에 기여했을 것으로 예상되는가?
- 성능이 개선되지 않았다면 어떤 선택을 다시 검토해야 하는가?


In [ ]:
comparison = pd.concat(
    [baseline_result, final_result],
    ignore_index=True
)

comparison["MAE_Change"] = comparison["MAE"] - baseline_result.loc[0, "MAE"]
comparison["RMSE_Change"] = comparison["RMSE"] - baseline_result.loc[0, "RMSE"]
comparison["Adjusted_R2_Change"] = (
    comparison["Adjusted_R2"]
    - baseline_result.loc[0, "Adjusted_R2"]
)

display(comparison.round(3))


In [ ]:
metric_comparison = comparison.set_index("Model")[[
    "MAE",
    "RMSE",
    "Adjusted_R2"
]].T

display(metric_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metric_comparison.loc["MAE"].plot(kind="bar", ax=axes[0])
axes[0].set_title("MAE Comparison")
axes[0].set_ylabel("Lower is Better")
axes[0].tick_params(axis="x", rotation=0)

metric_comparison.loc["RMSE"].plot(kind="bar", ax=axes[1])
axes[1].set_title("RMSE Comparison")
axes[1].set_ylabel("Lower is Better")
axes[1].tick_params(axis="x", rotation=0)

metric_comparison.loc["Adjusted_R2"].plot(kind="bar", ax=axes[2])
axes[2].set_title("Adjusted R² Comparison")
axes[2].set_ylabel("Higher is Better")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()
